Appendix D: Adding Bells and Whistles to the Training Loop

In [ ]:
# ========== 中文注释 ==========
# 单元作用：完成本附录的准备工作——导入基础库和 GPTModel，定义 GPT-124M 的
# 超参数配置，根据当前硬件自动选择计算设备(CUDA/MPS/CPU)，并实例化模型。
# 输入输出：本单元不涉及张量运算，主要产出全局变量 GPT_CONFIG_124M(dict)、
# device(torch.device)、model(GPTModel 实例，处于 eval 模式)。
from importlib.metadata import version
import torch

print("torch version:", version("torch"))


from previous_chapters import GPTModel
# 说明：previous_chapters.py 是与本 notebook 同目录下的本地脚本，汇总了
# 前面章节实现的 GPTModel 等组件；若本地没有该文件，可改用下面注释掉的
# llms_from_scratch 包导入方式(pip 安装后可用)。
# If the `previous_chapters.py` file is not available locally,
# you can import it from the `llms-from-scratch` PyPI package.
# For details, see: https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg
# E.g.,
# from llms_from_scratch.ch04 import GPTModel

# GPT-124M 的超参数配置(与前几章一致，这里把 context_length 从 1024
# 缩短为 256，是为了让本附录的训练循环演示跑得更快)
GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Vocabulary size
    "context_length": 256, # Shortened context length (orig: 1024)
    "emb_dim": 768,        # Embedding dimension
    "n_heads": 12,         # Number of attention heads
    "n_layers": 12,        # Number of layers
    "drop_rate": 0.1,      # Dropout rate
    "qkv_bias": False      # Query-key-value bias
}

# 依次检测 CUDA -> Apple Silicon 的 MPS -> CPU，自动选用可用的计算设备
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # Use PyTorch 2.9 or newer for stable mps results
    # 仅当 PyTorch 版本 >= 2.9 时才认为 MPS 后端足够稳定，否则退回 CPU
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
else:
    device = torch.device("cpu")

print("Device:", device)

# 固定随机种子以保证权重初始化可复现；随后实例化模型并切换到 eval 模式
# (eval 模式会关闭 dropout，这里只是先创建模型，后面训练前会再切回 train 模式)
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval();  # Disable dropout during inference


In [ ]:
# ========== 中文注释 ==========
# 单元作用：准备训练用的原始文本语料——若本地不存在 the-verdict.txt 则从
# GitHub 下载，否则直接读取本地文件；最终得到字符串变量 text_data。
# 下方三引号包裹的代码块是书中原始的 urllib 实现，因为在某些 VPN 环境下
# urllib 的旧协议设置可能连接失败，所以改用更稳健的 requests 库，这段
# urllib 代码被注释掉仅作参考，不会被执行。
import os
import requests

file_path = "the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

if not os.path.exists(file_path):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    text_data = response.text
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()

# The book originally used the following code below
# However, urllib uses older protocol settings that
# can cause problems for some readers using a VPN.
# The `requests` version above is more robust
# in that regard.

"""
import os
import urllib.request

if not os.path.exists(file_path):
    with urllib.request.urlopen(url) as response:
        text_data = response.read().decode('utf-8')
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()
"""

In [ ]:
# ========== 中文注释 ==========
# 单元作用：把上一单元读到的 text_data 按 9:1 切分为训练集/验证集，并分别
# 构造 PyTorch DataLoader。
# 张量形状：create_dataloader_v1 内部会用 GPT-2 分词器把文本切成 token id
# 序列，再用滑动窗口切成长度为 context_length(=256) 的样本；每个 batch
# 产出的 input_batch/target_batch 形状均为 [batch_size=2, context_length=256]，
# target 是 input 整体右移一位的下一 token 预测目标。
# 训练集用 stride=context_length(不重叠切片)+shuffle=True+drop_last=True，
# 验证集不打乱、不丢弃最后一个不满的 batch，便于稳定评估。
from previous_chapters import create_dataloader_v1
# Alternatively:
# from llms_from_scratch.ch02 import create_dataloader_v1


# Train/validation ratio
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))


torch.manual_seed(123)

train_loader = create_dataloader_v1(
    text_data[:split_idx],
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    text_data[split_idx:],
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

D.1 Learning rate warmup

In [ ]:
# ========== 中文注释 ==========
# 单元作用：定义学习率预热(warmup)相关的三个基础超参数。
# n_epochs：训练轮数；initial_lr：预热阶段起始学习率(很小，避免训练初期
# 梯度/参数尚未稳定时用大学习率导致震荡或发散)；peak_lr：预热结束后要
# 达到的峰值学习率。
n_epochs = 15
initial_lr = 0.0001
peak_lr = 0.01

In [ ]:
# ========== 中文注释 ==========
# 单元作用：计算整个训练过程的总迭代步数 total_steps(= 每轮的 batch 数 x
# 轮数)，以及预热阶段占用的步数 warmup_steps(取总步数的 20%)。
# 预热步数越多，学习率从 initial_lr 爬升到 peak_lr 的过程越平缓。
total_steps = len(train_loader) * n_epochs
warmup_steps = int(0.2 * total_steps) # 20% warmup
print(warmup_steps)

In [ ]:
# ========== 中文注释 ==========
# 单元作用：用一个"空跑"的训练循环演示线性学习率预热(linear warmup)的
# 效果——循环里没有真正计算损失/反向传播(注释掉了"Calculate loss and
# update weights")，只是逐步更新并记录每一步的学习率 track_lrs，最后
# 画出学习率随训练步数变化的曲线。
# 学习率调度要点：
#   - lr_increment = (peak_lr - initial_lr) / warmup_steps，即每一步
#     线性增加的量；
#   - 当 global_step < warmup_steps 时，lr 从 initial_lr 线性增长到
#     peak_lr；预热结束后 lr 恒定为 peak_lr(还没有加入余弦退火)。
#   - 通过遍历 optimizer.param_groups 手动把新学习率写回优化器，这是
#     PyTorch 里实现自定义学习率调度的常见写法。
lr_increment = (peak_lr - initial_lr) / warmup_steps

global_step = -1
track_lrs = []

optimizer = torch.optim.AdamW(model.parameters(), weight_decay=0.1)

for epoch in range(n_epochs):
    for input_batch, target_batch in train_loader:
        optimizer.zero_grad()
        global_step += 1

        if global_step < warmup_steps:
            lr = initial_lr + global_step * lr_increment
        else:
            lr = peak_lr

        # Apply the calculated learning rate to the optimizer
        for param_group in optimizer.param_groups:
            param_group["lr"] = lr
        track_lrs.append(optimizer.param_groups[0]["lr"])

        # Calculate loss and update weights
        # ...
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 3))
plt.ylabel("Learning rate")
plt.xlabel("Step")
total_training_steps = len(train_loader) * n_epochs
plt.plot(range(total_training_steps), track_lrs)
plt.tight_layout(); plt.savefig("1.pdf")
plt.show()

In [ ]:
# ========== 中文注释 ==========
# 单元作用：在线性预热的基础上加入"余弦退火"(cosine annealing)，构成
# 完整的 warmup + cosine decay 学习率调度，同样以"空跑"循环记录学习率
# 并画图对比。
# 学习率调度要点：
#   - min_lr = 0.1 * initial_lr，作为余弦退火阶段的下界(学习率不会降到 0，
#     而是降到一个很小的正值，有利于训练末期继续微调)；
#   - global_step < warmup_steps 时仍是线性预热(与上一单元相同)；
#   - 预热结束后，用 progress ∈ [0, 1] 表示"退火进度"，
#     lr = min_lr + (peak_lr - min_lr) * 0.5 * (1 + cos(pi * progress))，
#     这是标准的余弦退火公式：progress=0 时 lr=peak_lr，progress=1 时
#     lr=min_lr，中间按余弦曲线平滑下降(比线性衰减更平滑，末期下降更慢)。
import math

min_lr = 0.1 * initial_lr
track_lrs = []

lr_increment = (peak_lr - initial_lr) / warmup_steps
global_step = -1

for epoch in range(n_epochs):
    for input_batch, target_batch in train_loader:
        optimizer.zero_grad()
        global_step += 1

        # Adjust the learning rate based on the current phase (warmup or cosine annealing)
        if global_step < warmup_steps:
            # Linear warmup
            lr = initial_lr + global_step * lr_increment
        else:
            # Cosine annealing after warmup
            progress = ((global_step - warmup_steps) /
                        (total_training_steps - warmup_steps))
            lr = min_lr + (peak_lr - min_lr) * 0.5 * (
                1 + math.cos(math.pi * progress))

        # Apply the calculated learning rate to the optimizer
        for param_group in optimizer.param_groups:
            param_group["lr"] = lr
        track_lrs.append(optimizer.param_groups[0]["lr"])

        # Calculate loss and update weights
plt.figure(figsize=(5, 3))
plt.ylabel("Learning rate")
plt.xlabel("Step")
plt.plot(range(total_training_steps), track_lrs)
plt.tight_layout(); plt.savefig("2.pdf")
plt.show()

D.3 Gradient clipping

In [ ]:
# ========== 中文注释 ==========
# 单元作用：为演示梯度裁剪，先重新初始化一个模型(固定随机种子保证可复现)，
# 对一个 batch 计算损失并反向传播，从而让 model.parameters() 上产生梯度，
# 供后续单元检查/裁剪。
# 张量形状：input_batch/target_batch 取自上面 dataloader 循环里最后一次
# 迭代残留的变量，形状为 [batch_size=2, context_length=256]；loss 是标量。
from previous_chapters import calc_loss_batch
# Alternatively:
# from llms_from_scratch.ch05 import calc_loss_batch


torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.to(device)

loss = calc_loss_batch(input_batch, target_batch, model, device)
loss.backward()

In [ ]:
# ========== 中文注释 ==========
# 单元作用：定义辅助函数 find_highest_gradient，遍历模型所有参数的
# .grad，把每个参数梯度张量展平后取最大值，再取全局最大值，用来直观
# 观察"裁剪前"梯度里是否存在异常偏大的数值(梯度爆炸的信号)。
def find_highest_gradient(model):
    max_grad = None
    for param in model.parameters():
        if param.grad is not None:
            grad_values = param.grad.data.flatten()
            max_grad_param = grad_values.max()
            if max_grad is None or max_grad_param > max_grad:
                max_grad = max_grad_param
    return max_grad

print(find_highest_gradient(model))

In [ ]:
# ========== 中文注释 ==========
# 单元作用：对模型参数的梯度执行梯度裁剪(gradient clipping)，再次打印
# 最大梯度值，与上一单元的输出对比，验证裁剪效果。
# 梯度裁剪要点：torch.nn.utils.clip_grad_norm_ 会先计算所有参数梯度拼在
# 一起的整体 L2 范数(默认 norm_type=2)，如果这个整体范数超过 max_norm=1.0，
# 就把每个参数的梯度按同一比例缩小，使裁剪后的整体范数恰好等于 max_norm；
# 如果本来就没超过，则不做任何改动。这是按"全局范数"裁剪，而不是逐元素
# 裁剪，能防止梯度爆炸导致的参数更新过大/训练发散，同时保留梯度方向。
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
print(find_highest_gradient(model))

D.4 The modified training function

In [ ]:
# ========== 中文注释 ==========
# 单元作用：定义整合了"学习率预热 + 余弦退火 + 梯度裁剪 + 周期性评估 +
# 样本生成"的完整训练函数 train_model，并在下方实际调用它来训练模型。
# 这是本附录的核心单元，把前面几个单元里分别演示的技巧组合进一个可复用
# 的训练循环。
# 张量形状：input_batch/target_batch 来自 train_loader，形状均为
# [batch_size=2, context_length=256]；loss 为标量；calc_loss_batch 内部
# 会做一次前向传播(logits 形状约为 [batch_size, context_length,
# vocab_size])并与 target 计算交叉熵。
# 学习率调度要点(与前面单元一致)：
#   - global_step < warmup_steps：线性预热，从 initial_lr 增长到 peak_lr
#     (peak_lr 取自传入 optimizer 的初始学习率)；
#   - global_step >= warmup_steps：余弦退火，从 peak_lr 平滑下降到 min_lr。
# 梯度裁剪要点：
#   - 只有过了预热阶段才做梯度裁剪(预热阶段学习率还很小，梯度通常不会
#     爆炸，没必要裁剪)；
#   - ORIG_BOOK_VERSION=True 对应书中最初的写法 `global_step > warmup_steps`，
#     这会导致预热结束后紧接着的那一步(global_step == warmup_steps)漏做
#     裁剪；下面 else 分支用 `global_step >= warmup_steps` 修正了这个
#     "跳过一步裁剪"的小 bug，这是原作者自己在注释里说明并保留两种写法、
#     用 ORIG_BOOK_VERSION 开关切换，此处仅作说明，不需要再次修改。
from previous_chapters import evaluate_model, generate_and_print_sample
# Alternatively:
# 说明：原书注释中的函数名写成了 generate_and_print_samplee(多了一个字母
# e)，这是注释文本里的拼写错误(该行整体是注释，不影响实际代码执行)；此处
# 已按正确的函数名 generate_and_print_sample 更正，供读者需要时参考。
# from llms_from_scratch.ch05 import evaluate_model, generate_and_print_sample


ORIG_BOOK_VERSION = False


def train_model(model, train_loader, val_loader, optimizer, device,
                n_epochs, eval_freq, eval_iter, start_context, tokenizer,
                warmup_steps, initial_lr=3e-05, min_lr=1e-6):

    train_losses, val_losses, track_tokens_seen, track_lrs = [], [], [], []
    tokens_seen, global_step = 0, -1

    # Retrieve the maximum learning rate from the optimizer
    peak_lr = optimizer.param_groups[0]["lr"]

    # Calculate the total number of iterations in the training process
    total_training_steps = len(train_loader) * n_epochs

    # Calculate the learning rate increment during the warmup phase
    lr_increment = (peak_lr - initial_lr) / warmup_steps

    for epoch in range(n_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            global_step += 1

            # Adjust the learning rate based on the current phase (warmup or cosine annealing)
            if global_step < warmup_steps:
                # Linear warmup
                lr = initial_lr + global_step * lr_increment
            else:
                # Cosine annealing after warmup
                progress = ((global_step - warmup_steps) /
                            (total_training_steps - warmup_steps))
                lr = min_lr + (peak_lr - min_lr) * 0.5 * (1 + math.cos(math.pi * progress))

            # Apply the calculated learning rate to the optimizer
            for param_group in optimizer.param_groups:
                param_group["lr"] = lr
            track_lrs.append(lr)  # Store the current learning rate

            # Calculate and backpropagate the loss
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()

            # Apply gradient clipping after the warmup phase to avoid exploding gradients
            if ORIG_BOOK_VERSION:
                if global_step > warmup_steps:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            else:
                if global_step >= warmup_steps:  # the book originally used global_step > warmup_steps, which led to a skipped clipping step after warmup
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            tokens_seen += input_batch.numel()

            # Periodically evaluate the model on the training and validation sets
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader,
                    device, eval_iter
                )
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                # Print the current losses
                print(f"Ep {epoch+1} (Iter {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, "
                      f"Val loss {val_loss:.3f}"
                )

        # Generate and print a sample from the model to monitor progress
        generate_and_print_sample(
            model, tokenizer, device, start_context
        )

    return train_losses, val_losses, track_tokens_seen, track_lrs
import tiktoken

# Note:
# Uncomment the following code to calculate the execution time
# import time
# start_time = time.time()

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.to(device)

peak_lr = 0.001  # this was originally set to 5e-4 in the book by mistake
optimizer = torch.optim.AdamW(model.parameters(), lr=peak_lr, weight_decay=0.1)  # the book accidentally omitted the lr assignment
tokenizer = tiktoken.get_encoding("gpt2")

n_epochs = 15
train_losses, val_losses, tokens_seen, lrs = train_model(
    model, train_loader, val_loader, optimizer, device, n_epochs=n_epochs,
    eval_freq=5, eval_iter=1, start_context="Every effort moves you",
    tokenizer=tokenizer, warmup_steps=warmup_steps,
    initial_lr=1e-5, min_lr=1e-5
)

# Note:
# Uncomment the following code to show the execution time
# end_time = time.time()
# execution_time_minutes = (end_time - start_time) / 60
# print(f"Training completed in {execution_time_minutes:.2f} minutes.")

In [ ]:
# ========== 中文注释 ==========
# 单元作用：绘制上一单元 train_model 训练过程中记录下来的学习率变化曲线
# (lrs 列表，长度等于总的优化步数 total_training_steps)，可以直观看到
# "线性预热 + 余弦退火"的完整形状：先线性上升到峰值，再按余弦曲线平滑
# 下降。
plt.figure(figsize=(5, 3))
plt.plot(range(len(lrs)), lrs)
plt.ylabel("Learning rate")
plt.xlabel("Steps")
plt.show()

In [ ]:
# ========== 中文注释 ==========
# 单元作用：绘制训练过程中记录的训练/验证损失曲线(train_losses、
# val_losses)随"已见 token 数"(tokens_seen)的变化，并保存为 3.pdf。
# epochs_tensor 用 torch.linspace 在 [1, n_epochs] 区间内等间隔生成
# len(train_losses) 个点，仅用于给横轴/图例提供"大致处于第几个 epoch"
# 的参考刻度，并不是真实的每步 epoch 编号。
from previous_chapters import plot_losses
# Alternatively:
# from llms_from_scratch.ch05 import plot_losses


epochs_tensor = torch.linspace(1, n_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)
plt.tight_layout(); plt.savefig("3.pdf")
plt.show()